# 02. Modelo

## Librerias

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelBinarizer
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, mean_absolute_error
)
from imblearn.over_sampling import SMOTE




## Carga del dataset limpio

In [3]:
dt_quejas = pd.read_csv('..\data\dt_quejas_corr.csv')
dt_quejas.head(20)


<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Jon\AppData\Local\Temp\ipykernel_26880\1742754170.py:1: SyntaxWarning: invalid escape sequence '\d'
  dt_quejas = pd.read_csv('..\data\dt_quejas_corr.csv')


,Complaint ID,Product,Sub-product,Issue,State,ZIP code,Date received,Date sent to company,Company,Company response,Timely response?,Consumer disputed?,Company_grouped_filtered,Difference in days
0,1291006,Debt collection,Credit card,Communication tactics,TX,76119,2015-03-19,2015-03-19,"Premium Asset Services, LLC",In progress,Yes,Pending,Low Count Companies,0
1,1290580,Debt collection,Medical,Cont'd attempts collect debt not owed,TX,77479,2015-03-19,2015-03-19,Accounts Receivable Consultants Inc.,Closed with explanation,Yes,No,Low Count Companies,0
2,1290564,Mortgage,FHA mortgage,"Application, originator, mortgage broker",MA,02127,2015-03-19,2015-03-19,RBS Citizens,Closed with explanation,Yes,Yes,RBS Citizens,0
3,1291615,Credit card,Unknown,Other,CA,92592,2015-03-19,2015-03-19,Navy FCU,In progress,Yes,Pending,Navy FCU,0
4,1292165,Debt collection,Non-federal student loan,Cont'd attempts collect debt not owed,OH,43068,2015-03-19,2015-03-19,Transworld Systems Inc.,In progress,Yes,Pending,Transworld Systems Inc.,0
5,1291176,Debt collection,Payday loan,Communication tactics,OH,43068,2015-03-19,2015-03-19,ACE Cash Express Inc.,In progress,Yes,Pending,ACE Cash Express Inc.,0
6,1288848,Consumer loan,Installment loan,Managing the loan or lease,OH,44241,2015-03-18,2015-03-18,"CashCall, Inc.",Closed with explanation,Yes,Yes,"CashCall, Inc.",0
7,1288788,Debt collection,Payday loan,Communication tactics,CA,95124,2015-03-18,2015-03-18,ACE Cash Express Inc.,Closed with explanation,Yes,No,ACE Cash Express Inc.,0
8,1288324,Debt collection,"Other (phone, health club, etc.)",Cont'd attempts collect debt not owed,NJ,07067,2015-03-18,2015-03-18,"Credit Protection Association, L.P.",Closed with non-monetary relief,Yes,No,Low Count Companies,0
9,1288304,Debt collection,Payday loan,Taking/threatening an illegal action,TX,77433,2015-03-18,2015-03-18,Cottonwood Financial Ltd.,Closed with explanation,Yes,Yes,Low Count Companies,0


## Modelos a realizar

### 01. Modelo predictivo - ¿El consumidor disputará la queja?


In [4]:
y = dt_quejas['Consumer disputed?']

common_features = [
    'Product',
    'Issue',
    'Company response',
    'Timely response?',
    'Company_grouped_filtered',
    'State',
    'Difference in days'
]

features_with_subproduct = common_features + ['Sub-product']
features_without_subproduct = common_features.copy()

X_with_subproduct = dt_quejas[features_with_subproduct]
X_without_subproduct = dt_quejas[features_without_subproduct]

#### Modelo con Random Forest

In [5]:
def entrenar_modelo(X, y, labels):
    categorical_features = X.select_dtypes(include='object').columns.tolist()
    numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=42
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ],
        remainder='passthrough'
    )

    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
    ])

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)

    print(" Classification Report:\n", classification_report(y_test, y_pred))
    print(" Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    print("\n Métricas adicionales:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision (macro): {precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Recall (macro): {recall_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (macro): {f1_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")

    class_mapping = {label: i for i, label in enumerate(labels)}
    y_test_num = y_test.map(class_mapping)
    y_pred_num = pd.Series(y_pred).map(class_mapping)
    print(f"MAE: {mean_absolute_error(y_test_num, y_pred_num):.4f}")

    lb = LabelBinarizer()
    lb.fit(y_test)
    y_test_bin = lb.transform(y_test)

    if y_test_bin.shape[1] == 1:
        print("ROC AUC: sólo hay una clase activa, no se puede calcular.")
    else:
        auc = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')
        print(f"ROC AUC (OvR): {auc:.4f}")

    return clf


In [6]:
modelo_con_sub = entrenar_modelo(X_with_subproduct, y, labels=['No', 'Pending', 'Yes'])
modelo_sin_subproduct = entrenar_modelo(X_without_subproduct, y,labels=['No', 'Pending', 'Yes'])


 Classification Report:
               precision    recall  f1-score   support

          No       0.82      0.89      0.85      4082
     Pending       1.00      1.00      1.00       608
         Yes       0.24      0.15      0.19       942

    accuracy                           0.78      5632
   macro avg       0.69      0.68      0.68      5632
weighted avg       0.74      0.78      0.76      5632

 Confusion Matrix:
 [[3625    0  457]
 [   0  608    0]
 [ 797    0  145]]

 Métricas adicionales:
Accuracy: 0.7773
Precision (macro): 0.6869
Recall (macro): 0.6807
F1-score (macro): 0.6801
F1-score (weighted): 0.7573
MAE: 0.4453
ROC AUC (OvR): 0.7880
 Classification Report:
               precision    recall  f1-score   support

          No       0.82      0.86      0.84      4082
     Pending       1.00      1.00      1.00       608
         Yes       0.24      0.19      0.21       942

    accuracy                           0.76      5632
   macro avg       0.69      0.68      0.68  

El modelo de Random Forest que incluye a sub-product, obtuvo un accuracy del 77.7%, acompañado de un F1-score ponderado de 0.7573. En cuanto al F1-score macro, que otorga igual importancia a cada clase, se situó en 0.6801, lo que indica un rendimiento aceptable considerando el desbalance entre clases. La clase "No" (que el consumidor no disputó la queja) fue correctamente identificada con una precisión del 82% y una sensibilidad del 89%, mientras que la clase "Pending" (proceso aún en curso) fue perfectamente clasificada. Sin embargo, el desempeño para la clase "Yes" (quejas que fueron disputadas por el consumidor) fue considerablemente menor, con un F1-score de solo 0.19, reflejando una dificultad clara del modelo para identificar estos casos. El MAE (Mean Absolute Error) fue de 0.445, lo que representa un nivel intermedio de error categórico. Por su parte, el ROC AUC multiclase alcanzó un valor de 0.788, lo que muestra una buena capacidad del modelo para discriminar entre clases incluso en contextos multiclase complejos.

En este segundo modelo se eliminó la variable Sub-product, y los resultados se mantuvieron en una línea similar aunque ligeramente inferiores. El accuracy bajó a 76.1% y el F1-score ponderado a 0.7513, lo que sugiere una leve pérdida de rendimiento general. De nuevo, las clases "No" y "Pending" fueron clasificadas correctamente, con métricas prácticamente idénticas. En la clase "Yes", el modelo logró un F1-score algo superior (0.21), pero aún muy bajo para ser considerado útil de forma aislada. El MAE aumentó ligeramente a 0.478, reflejando una mayor proporción de errores. El ROC AUC también descendió un poco hasta 0.7847.

Aambos modelos enfrentan una limitación común importante: la incapacidad para predecir correctamente las disputas ("Yes"), probablemente debido al desbalance de clases y la posible falta de señales predictivas claras para este comportamiento.A pesar de ello, las métricas globales como el ROC AUC y el F1 ponderado indican que el modelo con Sub-product es ligeramente superior y, por tanto, sería el más recomendable de los dos para continuar con el proceso de desarrollo o despliegue.

##### Oversampling de la clase mayoritaria

In [7]:
def entrenar_modelo_con_smote(X, y, labels):

    categorical_features = X.select_dtypes(include='object').columns.tolist()
    numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=42
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ],
        remainder='passthrough'
    )

    X_train_enc = preprocessor.fit_transform(X_train)
    X_test_enc = preprocessor.transform(X_test)

    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_train_enc, y_train)

    clf = RandomForestClassifier(class_weight='balanced', random_state=42)
    clf.fit(X_resampled, y_resampled)

    y_pred = clf.predict(X_test_enc)
    y_proba = clf.predict_proba(X_test_enc)

    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    print("\n Métricas adicionales:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision (macro): {precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Recall (macro): {recall_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (macro): {f1_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")

    class_mapping = {label: i for i, label in enumerate(labels)}
    y_test_num = y_test.map(class_mapping)
    y_pred_num = pd.Series(y_pred).map(class_mapping)
    print(f"MAE: {mean_absolute_error(y_test_num, y_pred_num):.4f}")

    lb = LabelBinarizer()
    lb.fit(y_test)
    y_test_bin = lb.transform(y_test)

    if y_test_bin.shape[1] == 1:
        print("ROC AUC: sólo hay una clase activa, no se puede calcular.")
    else:
        auc = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')
        print(f"ROC AUC (OvR): {auc:.4f}")

    return clf

In [8]:
modelo_con_sub = entrenar_modelo_con_smote(X_with_subproduct, y, labels=['No', 'Pending', 'Yes'])
modelo_sin_subproduct = entrenar_modelo_con_smote(X_without_subproduct, y,labels=['No', 'Pending', 'Yes'])

Classification Report:
               precision    recall  f1-score   support

          No       0.82      0.91      0.86      4082
     Pending       1.00      1.00      1.00       608
         Yes       0.25      0.13      0.17       942

    accuracy                           0.79      5632
   macro avg       0.69      0.68      0.68      5632
weighted avg       0.74      0.79      0.76      5632

Confusion Matrix:
 [[3723    0  359]
 [   0  608    0]
 [ 821    0  121]]

 Métricas adicionales:
Accuracy: 0.7905
Precision (macro): 0.6905
Recall (macro): 0.6802
F1-score (macro): 0.6778
F1-score (weighted): 0.7621
MAE: 0.4190
ROC AUC (OvR): 0.7886
Classification Report:
               precision    recall  f1-score   support

          No       0.82      0.89      0.86      4082
     Pending       1.00      1.00      1.00       608
         Yes       0.24      0.15      0.18       942

    accuracy                           0.78      5632
   macro avg       0.69      0.68      0.68     

Se han entrenado dos modelos de clasificación para predecir si un consumidor disputará una queja, esta vez incluyendo un enfoque de balanceo de clases mediante SMOTE (Synthetic Minority Oversampling Technique). El objetivo era mejorar la capacidad del modelo para detectar correctamente la clase minoritaria: "Yes". Aunque el uso de SMOTE ha permitido una ligera mejora en métricas generales como el accuracy y el MAE, no ha logrado mejorar de forma significativa el rendimiento sobre la clase minoritaria "Yes". La precisión y el F1-score se mantuvieron prácticamente iguales, y el recall incluso disminuyó ligeramente.

##### Hiperparametrizacion con y sin SMOTE

In [10]:
def entrenar_modelo_gs(X, y, labels):
    categorical_features = X.select_dtypes(include='object').columns.tolist()
    numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=42
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ],
        remainder='passthrough'
    )

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42))
    ])

    param_grid = {
        'classifier__n_estimators': [100],
        'classifier__max_depth': [None, 10],
        'classifier__min_samples_split': [2],
        'classifier__min_samples_leaf': [1],
        'classifier__class_weight': ['balanced']
    }

    grid_search = GridSearchCV(
        pipeline,
        param_grid,
        cv=3,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_

    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)

    print(" Classification Report:\n", classification_report(y_test, y_pred))
    print(" Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    print("\n Métricas adicionales:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision (macro): {precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Recall (macro): {recall_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (macro): {f1_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")

    class_mapping = {label: i for i, label in enumerate(labels)}
    y_test_num = y_test.map(class_mapping)
    y_pred_num = pd.Series(y_pred).map(class_mapping)
    print(f"MAE: {mean_absolute_error(y_test_num, y_pred_num):.4f}")

    lb = LabelBinarizer()
    lb.fit(y_test)
    y_test_bin = lb.transform(y_test)

    if y_test_bin.shape[1] == 1:
        print("ROC AUC: solo hay una clase activa, no se puede calcular.")
    else:
        auc = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')
        print(f"ROC AUC (OvR): {auc:.4f}")

    print(f"\n✅ Mejores parámetros encontrados: {grid_search.best_params_}")

    return best_model


In [11]:
modelo_gs = entrenar_modelo_gs(X_with_subproduct, y, labels=['No', 'Pending', 'Yes'])

Fitting 3 folds for each of 2 candidates, totalling 6 fits
 Classification Report:
               precision    recall  f1-score   support

          No       0.82      0.89      0.85      4082
     Pending       1.00      1.00      1.00       608
         Yes       0.24      0.15      0.19       942

    accuracy                           0.78      5632
   macro avg       0.69      0.68      0.68      5632
weighted avg       0.74      0.78      0.76      5632

 Confusion Matrix:
 [[3625    0  457]
 [   0  608    0]
 [ 797    0  145]]

 Métricas adicionales:
Accuracy: 0.7773
Precision (macro): 0.6869
Recall (macro): 0.6807
F1-score (macro): 0.6801
F1-score (weighted): 0.7573
MAE: 0.4453
ROC AUC (OvR): 0.7880

✅ Mejores parámetros encontrados: {'classifier__class_weight': 'balanced', 'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}
